# Welcome to the RandQuantKry Tutorial Part 3
 
This is the third tutorial for **RandQuantKry** on how to benchmark. <a href="tutorial.ipynb"><b>Part 1</b></a>, on how to set up a hamiltonian, an initial state and run a time evolution, along with <a href="tutorial_evolution_methods.ipynb"><b> Part 2</b></a> on how to call the different evolution methods and the <a href="tutorial_parameter_study.ipynb"><b>parameter study</b></a> tutorial can be found in the respectively named Jupyter Notebooks. Happy Benchmarking :)

In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
# these files can be found in src/ and are the main building blocks of this package
import numpy as np
from randquantkry.hamiltonian import build_hamiltonian
from randquantkry.benchmark import Benchmark
from randquantkry.post_processing import plot_runtime, plot_memory

# utils/ contains all helper functions that allow for example logging
from randquantkry.utils.logging_config import configure_logging

In [ ]:
configure_logging(console_level='INFO')

As always we start by setting up our hamiltonian and initial state

In [ ]:
n_qubits = 8
rng = np.random.default_rng(0)

terms = []
for i in range(n_qubits):
    terms.append(('X', i, float(rng.uniform(0.5, 1.5))))       # X field, every qubit
for i in range(n_qubits - 1):
    terms.append(('ZZ', i, float(rng.uniform(0.3, 0.8))))      # ZZ coupling, every neighboring pair

hamiltonian = build_hamiltonian(n_qubits, terms)

n_qubits = hamiltonian.num_qubits
initial_state = np.zeros(2**n_qubits, dtype=complex)
initial_state[128] = 1.0   # |1000...0>

The benchmarking function expects a benchmarking object consisting of a hamiltonian, initial state, krylov dimension, number of random vectors and optionally a seed, so let's initialize a benchmarking object:

In [ ]:
time = 1.0
krylov_dim=50
num_random_vectors=2*krylov_dim+1
seed = 67

benchmark1 = Benchmark(hamiltonian, initial_state, time, krylov_dim, num_random_vectors, seed = seed, output_dir="results/")

To call the actual function we require a list of qubits we want to compare along with a list of methods.

<div class="alert alert-block alert-warning">
<b>Please Note: </b> 

The naive method for a 10-12 qubit system only about 1 GB of RAM is required but 13-14 qubit systems quickly scale to 16GB of RAM therefore the naive method is bounded to 12 qubits. This bound can be adjusted in <a href="../src/evolution.py"><b>src/evolution.py</b></a> with the parameter: **MAX_NAIVE_QUBITS**. 

There are two ways to currently track the memory: "tracemalloc" and "peak_rss". The default is tracemalloc as it provides enough accuracy for this usecase, peak_rss can provide a bit more accuracy but requires a lot more runtime so use at your own risk. To use it simply change memory_type = "peak_rss".
</div>

In [ ]:
results, run_dir = Benchmark.benchmark(benchmark1, n_qubits_list=[6, 8, 10, 14, 16, 18], methods=['naive', 'krylov', 'randomized_krylov'], memory_type = "tracemalloc")

In [ ]:
print(f"Results written to {run_dir}")

The benchmarking runs each method for the given number of qubits and records the runtime and memory. Both can be plotted using deticated plotting functions that can be found in <a href="../src/post_processing.py"><b>src/post_processing.py</b></a>:

In [ ]:
plot_memory(results,ram_limits={'Typical PC RAM: 16GB': 16, 'Typical HPC RAM: 512GB': 512}, save_path= run_dir)

In [ ]:
plot_runtime(results, save_path= run_dir)